In [2]:
from langchain_ollama import OllamaLLM 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

chain = (
    PromptTemplate.from_template(
        """Given the user question below, classify it as either being about `Films`, `Cars`, or `Other`.

Do not respond with more than one word.

<question>
{question}
</question>

Classification:"""
    )
    | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")
    | StrOutputParser()
)

chain.invoke({"question": "Tell me how to clean my ford focus properly?"})

'Cars'

In [18]:
# Sub chains
# tools_chain = PromptTemplate.from_template(

film_chat_chain = PromptTemplate.from_template(
    """You are a friendly and knowledgeable assistant specializing in films. Your goal is to engage users in conversations to understand their film preferences, interests, and what aspects of films are most important to them. \
    Keep the conversation a reasonably short length, and ask questions to get more information from the user. \
    
    Respond to the following question:
    
    Question: {question}
    Answer:
    """
    ) | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")

explain_rec_chain = PromptTemplate.from_template(
    """You are an expert in films, and you are talking to a user. You need to briefly go through as to why a particular film was recommended to the user. 
    The recommendation score format is: 
    `Title (Release Date): total_score, cast_score, director_score, genre_score, collaborative_filtering_score`. 
    Given the following breakdown:
    
    {input_text}
    
    Provide a clear and concise explanation of the scores and why the film might be recommended.
    
    Answer:
    """
    ) | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")

glean_feed_back_chain = PromptTemplate.from_template(
    """
    Analyze the sentiment of the following sentence.
    
    Provide sentiment scores on the importance of these features, and then sentiment scores for items within these features:
    - Films
    - Actors
    - Directors
    - Genres
    Add a \n after each feature.
    
    Don't be verbose, just provide the scores with no other info.
    - Item ratings; Give the ratings of items within a feature, for example scores of various films.
    - Feature importance; the importance of the features to the user. E.g. Importance of actors to the user.
    
    Sentence: {input_text}
    """
    ) | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")


In [14]:
explain_rec_chain.invoke({"input_text": "Batman Begins,2005-06-10,12.571,5.40,4.88,7.4620,3.286"})

'Let\'s break down the scores for "Batman Begins" (2005):\n\n**Total Score: 12.571**\n\nThis total score represents an overall rating given to the film, with 12.57 being a weighted average of the individual scores.\n\n**Cast Score: 5.40**\n\nThe cast score indicates how well the actors performed in their roles. A high score like 5.40 suggests that the cast delivered strong performances, making it easier for them to win awards and gain recognition.\n\n**Director Score: 4.88**\n\nThe director score reflects the director\'s skill and vision behind the film. In this case, Christopher Nolan\'s direction is praised for its innovative storytelling, action sequences, and themes, resulting in a score of 4.88.\n\n**Genre Score: 7.4620**\n\nThe genre score assesses how well the film adheres to or subverts genre conventions. Batman Begins\' score of 7.46 indicates that it successfully blends elements of superhero origin stories with psychological thriller and action movie tropes, creating a unique

In [8]:
film_chat_chain.invoke({"question": "Hey, I like the movie 'The Dark Knight', I'm in between minds about what really defined it as a great movie."})

'"The Dark Knight" is an incredible film that has left a lasting impact on the superhero genre. What specifically do you think was missing or didn\'t quite come together for you to consider it truly great? Was it Heath Ledger\'s iconic performance, the themes of chaos and anarchy, or perhaps the way Christopher Nolan balanced action and drama?\n\nAlso, are you more interested in the technical aspects of filmmaking (e.g., cinematography, score), the performances, the storytelling, or something else that contributed to your impression of the movie?'

In [19]:
explain_rec_chain.invoke({"input_text": "Batman Begins,2005-06-10,12.571,5.40,4.88,7.4620,3.286"})

'Let\'s break down the recommendation scores for "Batman Begins" (2005):\n\n**Title (Release Date): Batman Begins (2005-06-10)**: This is the title of the film, including its release date.\n\n**total_score**: This score represents the overall recommendation score for the film. A total score of 12.571 indicates that the algorithm has highly recommended this film to you based on various factors.\n\n**cast_score**: This score reflects the quality of the cast in the film. In this case, it\'s worth noting that the cast includes Christian Bale, Liam Neeson, and Cillian Murphy, among others. A high cast score suggests that the actors delivered strong performances, which is likely to have contributed to the film\'s success.\n\n**director_score**: The director score highlights Christopher Nolan\'s direction, who brought a fresh perspective to the Batman franchise. His expertise in crafting engaging storylines and suspenseful scenes is evident in this film, making it a standout in the series.\n\

In [20]:
glean_feed_back_chain.invoke({"input_text": "I like the film bullet train. In my opinion, directors are more important in films, I don't think actors are important. I don't mind what genre it is tbh. I do think Paul Mescal is a good actor though."})

'Here are the sentiment scores:\n\n- Films\n  - Overall importance: 6/10\n  - Bullet Train rating: 7/10\n- Actors\n  - Overall importance: 4/10\n  - Paul Mescal rating: 8/10\n- Directors\n  - Overall importance: 8/10\n  - Importance to user: 9/10\n- Genres\n  - Overall importance: 3/10'